# Reproduction notebook: ESG / governance out-of-sample ROA prediction
This notebook regenerates the revision analyses requested by the reviewers from the licensed LSEG/Refinitiv extract (`Cleaned_GridExport_March_27_2023.xlsx`): the European sample reconstruction, the chronological split (Table E1), the naive persistence and pooled AR(1) benchmarks (Table 5 benchmark row, Figure 1), the Diebold-Mariano test, the held-out split-conformal recalibration (Figure 3), the complete-case re-estimation, and verification of the descriptive statistics in Table 2.

The raw vendor data cannot be redistributed; place your licensed extract next to this notebook. The tuned four-algorithm pipeline that produces Tables 5 to 14 uses the same preprocessing and temporal split shown here.

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

XL = 'Cleaned_GridExport_March_27_2023.xlsx'   # licensed LSEG/Refinitiv extract
SEED = 42
EU = ['Austria','Belgium','Denmark','Finland','France','Germany','Ireland; Republic of',
      'Italy','Luxembourg','Netherlands','Norway','Portugal','Spain','Sweden','Switzerland','United Kingdom']

## 1. European sample and long-form panel\nFilter to the 16 headquarters countries (373 firms) and reshape FY-12..FY-1 to firm-period rows; the outcome is one-period-ahead ROA. ROA fields are stored as decimal fractions.

In [ ]:
df = pd.read_excel(XL, sheet_name='Cleaned Data')
d = df[df['Country of Headquarters'].isin(EU)].copy().reset_index(drop=True)
print('European firms:', d['Identifier (RIC)'].nunique(), '| rows:', len(d))

gov = ['Board Gender Diversity, Percent Score','Executive Members Gender Diversity, Percent Score',
       'Board Size','Audit Committee Independence Score','Audit Committee Expertise Score',
       'Auditor Tenure Score','CEO-Chairman Separation Score','CSR Sustainability Committee Score']
esg = ['Environmental Pillar Score','Social Pillar Score','Governance Pillar Score']

recs = []
for _, r in d.iterrows():
    for k in range(12, 1, -1):                      # forecast origin FY-k
        out = r.get(f'ROA Total Assets, Percent FY-{k-1}')   # one-period-ahead ROA
        if pd.isna(out):
            continue
        ta, td = r.get(f'Total Assets, Reported (USD) FY-{k}'), r.get(f'Total Debt (USD) FY-{k}')
        mc = r.get(f'Company Market Capitalization (USD) FY-{k}')
        rec = {'ric': r['Identifier (RIC)'], 'fy_k': k, 'y': float(out),
               'roa_t': r.get(f'ROA Total Assets, Percent FY-{k}'),
               'ln_ta': np.log1p(float(ta)) if pd.notna(ta) else np.nan,
               'lev': (float(td)/float(ta)) if (pd.notna(td) and pd.notna(ta) and float(ta) != 0) else np.nan,
               'ln_mc': np.log1p(float(mc)) if pd.notna(mc) else np.nan,
               'country': r['Country of Headquarters'], 'industry': r['ICB Industry name nan']}
        for c in esg + gov:
            rec[c] = r.get(f'{c} FY-{k}')
        recs.append(rec)
m = pd.DataFrame(recs)
for c in esg + gov + ['roa_t']:
    m[c] = pd.to_numeric(m[c], errors='coerce')
print('measurable one-period-ahead outcomes:', len(m))   # 3,968

## 2. Chronological split (Table E1)

In [ ]:
counts = m.groupby('fy_k').size().sort_index(ascending=False)
cal = {k: 2024 - k for k in range(0, 13)}
for k in sorted(counts.index, reverse=True):
    role = 'train' if 4 <= k <= 12 else ('test' if k in (2,3) else 'n.a.')
    print(f'origin FY-{k} (~{cal[k]}) -> outcome FY-{k-1} (~{cal[k-1]}): {counts[k]:4d}  [{role}]')
train = m[m.fy_k.between(4,12)].copy()
test  = m[m.fy_k.isin([2,3])].copy()
print('train', len(train), '| test', len(test))

## 3. Naive persistence and pooled AR(1) benchmarks (Table 5 row, Figure 1)\nPersistence sets next-period ROA equal to current ROA. R-squared is reported against the sample mean and against persistence.

In [ ]:
def metrics(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    e = y - yhat
    return (np.mean(np.abs(e)), np.sqrt(np.mean(e**2)),
            1 - np.sum(e**2)/np.sum((y - y.mean())**2), np.sum(e**2))

tp = test.dropna(subset=['roa_t','y'])
mae_p, rmse_p, r2m_p, ss_p = metrics(tp['y'], tp['roa_t'])
print(f'Persistence  MAE={mae_p:.4f}  RMSE={rmse_p:.4f}  R2_vs_mean={r2m_p:.3f}  (n={len(tp)})')

b, a, rpear, _, _ = stats.linregress(train.dropna(subset=['roa_t','y'])['roa_t'],
                                     train.dropna(subset=['roa_t','y'])['y'])
pred_ar = a + b*tp['roa_t']
mae_ar, rmse_ar, r2m_ar, _ = metrics(tp['y'], pred_ar)
print(f'AR(1)        MAE={mae_ar:.4f}  RMSE={rmse_ar:.4f}  R2_vs_mean={r2m_ar:.3f}')

for name, rmse_model in [('Full ExtraTrees (paper)', 0.0645), ('Financial-only (paper)', 0.0649)]:
    ss_model = (rmse_model**2)*len(tp)
    print(f'{name}: R2 vs persistence = {1 - ss_model/ss_p:.3f}')

## 4. Nested ExtraTrees pipeline and Diebold-Mariano test\nWithin-pipeline median imputation + missingness-equivalent handling, standardization, one-hot country/industry, training-outcome winsorized at 0.5/99.5. Reproduces the reported headline metrics to within ~0.001 MAE.

In [ ]:
fin = ['roa_t','ln_ta','lev','ln_mc']; cat = ['country','industry']
sets = {'Financial only': fin, 'Financial + ESG': fin+esg, 'Full': fin+esg+gov}
def build(num):
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat)])
trw = train.copy(); lo, hi = trw.y.quantile([.005,.995]); trw['y'] = trw.y.clip(lo,hi)
pred = {}
for name, num in sets.items():
    pipe = Pipeline([('pre', build(num)), ('et', ExtraTreesRegressor(n_estimators=600, random_state=SEED, n_jobs=-1))]).fit(trw[num+cat], trw.y)
    pr = pipe.predict(test[num+cat]); pred[name] = pr
    print(f'{name:16s} MAE={np.mean(np.abs(test.y.values-pr)):.4f}  RMSE={np.sqrt(np.mean((test.y.values-pr)**2)):.4f}')
dl = np.abs(test.y.values-pred['Financial only']) - np.abs(test.y.values-pred['Full'])
t = dl.mean()/(dl.std(ddof=1)/np.sqrt(len(dl)))
print(f'Diebold-Mariano (MAE, financial vs full): t={t:.2f}, p={2*(1-stats.norm.cdf(abs(t))):.4f}')

## 5. Held-out split-conformal recalibration (Figure 3)\nFit on FY-12..FY-5, hold FY-4 out as a disjoint calibration set, evaluate coverage on FY-3/FY-2.

In [ ]:
num = fin+esg+gov
tr2 = m[m.fy_k.between(5,12)].copy(); lo,hi = tr2.y.quantile([.005,.995]); tr2['y']=tr2.y.clip(lo,hi)
cal_set = m[m.fy_k==4].copy()
pipe2 = Pipeline([('pre', build(num)), ('et', ExtraTreesRegressor(n_estimators=600, random_state=SEED, n_jobs=-1))]).fit(tr2[num+cat], tr2.y)
res = np.abs(cal_set.y.values - pipe2.predict(cal_set[num+cat])); n = len(res)
q = np.quantile(res, np.ceil(0.90*(n+1))/n, method='higher')
cov = np.mean(np.abs(test.y.values - pipe2.predict(test[num+cat])) <= q)
print(f'Held-out calibration: radius={q:.4f}  empirical coverage={cov:.3f}  width={2*q:.4f}  (nominal 0.90)')

## 6. Complete-case re-estimation (survivorship / imputation check)

In [ ]:
trc = trw.dropna(subset=num); tec = test.dropna(subset=num)
pc = Pipeline([('pre', build(num)), ('et', ExtraTreesRegressor(n_estimators=600, random_state=SEED, n_jobs=-1))]).fit(trc[num+cat], trc.y)
e = tec.y.values - pc.predict(tec[num+cat])
print(f'Complete-case Full: n_train={len(trc)} n_test={len(tec)}  MAE={np.mean(np.abs(e)):.4f}  RMSE={np.sqrt(np.mean(e**2)):.4f}')

## 7. Verification of Table 2 audit-committee expertise distribution

In [ ]:
vals = []
for _, r in d.iterrows():
    for k in range(12, 1, -1):
        o = r.get(f'ROA Total Assets, Percent FY-{k-1}'); v = r.get(f'Audit Committee Expertise Score FY-{k}')
        if pd.notna(o) and pd.notna(v):
            vals.append(float(v))
s = pd.Series(vals)
print(f'n={len(s)} mean={s.mean():.3f} sd={s.std():.3f} min={s.min():.3f} '
      f'p25={s.quantile(.25):.3f} median={s.median():.3f} p75={s.quantile(.75):.3f} max={s.max():.3f}')